# Phase 0A — Neural CDE Conditioning

Training and validation of the Neural CDE-conditioned diffusion model on KF12 using the validated RK4×4 solver.

## Setup

## Sync Github

In [2]:
import os
import subprocess

REPO_PATH = "/kaggle/working/diffusion-portfolio"

REPO_URL = (
    "https://github.com/"
    "Amir-Entezari/"
    "diffusion-portfolio.git"
)

if not os.path.exists(
    os.path.join(
        REPO_PATH,
        ".git",
    )
):
    subprocess.run(
        [
            "git",
            "clone",
            REPO_URL,
            REPO_PATH,
        ],
        check=True,
    )

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "fetch",
        "origin",
        "main",
    ],
    check=True,
)

subprocess.run(
    [
        "git",
        "-C",
        REPO_PATH,
        "reset",
        "--hard",
        "origin/main",
    ],
    check=True,
)

CURRENT_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        REPO_PATH,
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()

os.chdir(
    REPO_PATH
)

print(
    "Synced to latest main:"
)
print(
    CURRENT_COMMIT
)

subprocess.run(
    [
        "git",
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)

Cloning into '/kaggle/working/diffusion-portfolio'...


HEAD is now at 6cfa645 Organize Diffolio reproduction notebook
Synced to latest main:
6cfa645e93b3e84eb74f26bff0886226a6f7aa05
6cfa645 Organize Diffolio reproduction notebook


From https://github.com/Amir-Entezari/diffusion-portfolio
 * branch            main       -> FETCH_HEAD


CompletedProcess(args=['git', 'log', '-1', '--oneline'], returncode=0)

In [3]:
!pip install -q -e '.[cde]'

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 2.5 MB/s eta 0:00:00
  Building editable for diffusion-portfolio (pyproject.toml) ... done


## Backup

In [12]:
EXPERIMENT = "phase0/neural_cde_rk4x4"
RUN_DIR = "/kaggle/working/phase0_cde"

!python scripts/kaggle_artifacts.py backup \
    --handle amirentezari/diffusion-portfolio \
    --experiment {EXPERIMENT} \
    --source {RUN_DIR}

Creating new artifact store:
  handle: amirentezari/diffusion-portfolio
Uploading Dataset https://kaggle.com/datasets/amirentezari/diffusion-portfolio ...
Starting upload for file /kaggle/working/diffusion-portfolio-artifacts/.artifact_store.json
Uploading: 100%|█████████████████████████████████| 129/129 [00:00<00:00, 292B/s]
Upload successful: /kaggle/working/diffusion-portfolio-artifacts/.artifact_store.json (129B)
Starting upload for file /kaggle/working/diffusion-portfolio-artifacts/phase0/neural_cde_rk4x4/summary.json
Uploading: 100%|███████████████████████████████| 583/583 [00:00<00:00, 1.42kB/s]
Upload successful: /kaggle/working/diffusion-portfolio-artifacts/phase0/neural_cde_rk4x4/summary.json (583B)
Starting upload for file /kaggle/working/diffusion-portfolio-artifacts/phase0/neural_cde_rk4x4/best.pt
Uploading: 100%|███████████████████████████| 16.0M/16.0M [00:00<00:00, 30.2MB/s]
Upload successful: /kaggle/working/diffusion-portfolio-artifacts/phase0/neural_cde_rk4x4/best.pt 

In [13]:
!pytest -q \
    tests/unit/test_cde.py \
    tests/unit/test_conditional_diffusion_model.py

.........................                                                [100%]


## Training

In [ ]:
!python scripts/train_diffusion.py \
    --config configs/phase0_cde.yaml \
    --output-dir /kaggle/working/phase0_cde \
    --device cuda

In [14]:
!PYTHONPATH=/kaggle/working/diffusion-portfolio/src \
python scripts/evaluate_diffusion_validation.py \
    --run-dir /kaggle/working/phase0_cde \
    --batch-size 32 \
    --device cuda

DIFFUSION VALIDATION FORECAST EVALUATION
Device: cuda
GPU: Tesla T4
History encoder: cde
Scenarios per date: 256
Inference batch size: 32
Sampling seed: 12345

Validation samples: 1256
Validation start: 2000-01-03 00:00:00
Validation end: 2004-12-31 00:00:00

Checkpoint epoch: 1
Checkpoint validation loss: 1.8253416084939507

Generating VALIDATION scenarios...
Validation sampling: 100%|██████████████████████| 40/40 [02:34<00:00,  3.86s/it]

Saved validation scenarios: /kaggle/working/phase0_cde/validation_scenarios.npz
Scenario cube: (1256, 256, 12)
Raw scenario mean: 0.0002318994520464912
Raw scenario std: 0.008044425398111343
Raw scenario max abs: 0.05565167963504791

Computing VALIDATION probabilistic metrics...

VALIDATION PROBABILISTIC EVALUATION
CRPS mean: 0.008353461880032392
CRPS std across assets: 0.0021106179779315924
Energy Score: 0.034737111313670174

Calibration:
   level         PICP          ACE
    0.50     0.325571    -0.174429
    0.80     0.560576    -0.239424
    0.

In [16]:
import json
import pandas as pd
import torch

RUN_DIR = "/kaggle/working/phase0_cde"

summary = json.load(open(f"{RUN_DIR}/summary.json"))
history = pd.read_csv(f"{RUN_DIR}/history.csv")
checkpoint = torch.load(
    f"{RUN_DIR}/best.pt",
    map_location="cpu",
    weights_only=False,
)

print("epochs recorded:", len(history))
print("summary best:", summary["best_epoch"], summary["best_val_loss"])
print("checkpoint:", checkpoint["epoch"], checkpoint["val_loss"])
print(history.tail())

epochs recorded: 50
summary best: 48 1.42807903866859
checkpoint: 1 1.8253416084939507
    epoch  train_loss  val_loss
45     46    0.680125  1.452663
46     47    0.697422  1.452909
47     48    0.723848  1.428079
48     49    0.695568  1.444216
49     50    0.706177  1.445503
